# NFL Defensive Playcall — Causal Inference Pipeline

**Causal Question:** *What is the causal effect of a defensive playcall on EPA, given the game state, offensive personnel, and offensive formation?*

This notebook walks through the full pipeline end-to-end, one stage at a time. Each stage builds on the previous one, so cells should be run in order on first execution. After the initial run, individual stages can be re-run independently to experiment with hyperparameters, swap in new data, or modify the agent.

---

### Pipeline Overview

| Stage | Purpose |
|---|---|
| 0 | Install dependencies |
| 1 | Load & engineer features from nflverse |
| 2 | Register the causal DAG and check identification |
| 3a | Fit the propensity model |
| 3b | Fit the outcome model |
| 3c | AIPW doubly-robust ATE estimation |
| 3d | Causal Forest CATE estimation |
| 3e | Sensitivity analysis |
| 4 | Build the defensive coordinator agent |
| 5 | Run example playcall recommendations |

---
## Stage 0 — Install Dependencies

Run this cell once to install the required packages. Skip if your environment already has them.

- **nfl-data-py**: Python wrapper for the nflverse play-by-play dataset
- **dowhy**: Causal model registration, DAG-based identification, and refutation tests
- **econml**: Microsoft's library for causal ML — provides the Doubly Robust Learner and Causal Forest
- **lightgbm**: Gradient boosting framework used for both propensity and outcome models

In [ ]:
%pip install nfl-data-py pandas numpy scikit-learn lightgbm econml dowhy networkx matplotlib --quiet

---
## Stage 1 — Data Loading & Feature Engineering

### What this stage does
Loads raw play-by-play data from nflverse and transforms it into a modeling-ready dataframe. Every variable in the causal graph is operationalized here as a concrete column.

### Key design decisions

**Treatment construction:** The defensive playcall is a composite label formed from three observable dimensions available in nflverse — pressure tier (pass rushers), box density (defenders in box), and coverage shell (man vs zone). This produces a set of discrete treatment levels (e.g. `standard_standard_box_ZONE`) that each represent a coherent defensive strategy.

**Player quality proxies:** Proprietary player grades (e.g. PFF) are replaced with three publicly available proxies:
1. `def_team_avg_epa` — rolling team-season defensive EPA, absorbing unit-level quality differences
2. `qb_avg_epa` — QB season EPA, capturing offensive skill
3. Sensitivity analysis in Stage 3e will formally quantify how much hidden player quality would need to exist to overturn our conclusions

**Offensive tendencies:** Pass rate is computed as a rolling lagged mean per team × down × distance × formation. The lag prevents data leakage — we only use information the defense would have had *before* the snap.

> **Edit here:** Change `SEASONS` to adjust the training window. More seasons = more data but older play styles. 3 seasons is a reasonable balance.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from pipeline_01_data import (
    load_pbp, filter_plays, build_game_state, build_offensive_context,
    build_player_proxies, build_offensive_tendencies, build_treatment, build_model_df,
    ALL_CONFOUNDERS, TREATMENT_COL, OUTCOME_COL
)

# ── Configuration ─────────────────────────────────────────────────────────────
SEASONS = [2021, 2022, 2023]   # ← edit to change training window

print(f"Loading seasons: {SEASONS}")

In [ ]:
# Load raw play-by-play from nflverse
pbp = load_pbp(SEASONS)
print(f"Raw plays loaded: {len(pbp):,}")
print(f"Columns available: {list(pbp.columns[:20])} ...")

In [ ]:
# Filter to scrimmage plays and engineer all features
df = filter_plays(pbp)
df = build_game_state(df)
df = build_offensive_context(df)
df = build_player_proxies(df, roster_df=None)   # pass a roster DataFrame here if available
df = build_offensive_tendencies(df)
df, label_encoder = build_treatment(df)
df = build_model_df(df)

print(f"\nModeling dataset: {len(df):,} plays")
print(f"Treatment levels ({len(label_encoder.classes_)}): {list(label_encoder.classes_)}")
print(f"\nEPA summary:")
df[OUTCOME_COL].describe().round(3)

In [ ]:
# Quick sanity checks — inspect feature distributions
print("Formation distribution:")
print(df['formation'].value_counts().head(10))

print("\nDefensive playcall distribution:")
print(df['def_playcall'].value_counts())

print("\nPersonnel group distribution:")
print(df['personnel_group'].value_counts().head(8))

---
## Stage 2 — Causal DAG Registration & Identification

### What this stage does
Registers the structural causal model as a directed acyclic graph (DAG), then uses the **backdoor criterion** to formally verify that the causal effect of the defensive playcall on EPA is *identified* — i.e., estimable from observational data given our adjustment set.

### Theory: The Backdoor Criterion
A set of variables **Z** satisfies the backdoor criterion relative to (Treatment → Outcome) if:
1. No variable in **Z** is a descendant of Treatment
2. **Z** blocks every "backdoor path" from Treatment to Outcome (paths that flow against the arrow into Treatment)

If backdoor is satisfied, the causal effect is identified as:
$$P(Y \mid do(T=t)) = \sum_z P(Y \mid T=t, Z=z) \cdot P(Z=z)$$

Our adjustment set is `{game_state, off_personnel, off_formation, off_tendencies, player_proxies}`. Offensive playcall is **not** included — it sits downstream of formation and conditioning on it would open a collider path.

### Positivity Check
Identification also requires **positivity**: every defensive call must have a nonzero probability of being observed in every context stratum. We check this empirically and flag strata with sparse coverage — the agent will lower its confidence in these regions.

> **Edit here:** If DoWhy reports the effect is unidentified, check for missing edges in `CAUSAL_GRAPH_GML` in `02_causal_identification.py`, or add variables to `common_causes`.

In [ ]:
from pipeline_02_identification import (
    build_causal_model, identify_estimand, check_positivity, visualize_dag
)

# Render and save the DAG
visualize_dag(save_path='dag.png')

from IPython.display import Image
Image('dag.png')

In [ ]:
# Register the causal model and identify the estimand
causal_model = build_causal_model(df)
estimand = identify_estimand(causal_model)

# The estimand tells us exactly what quantity we're targeting
# and which identification strategy DoWhy will use (backdoor, IV, etc.)

In [ ]:
# Positivity check: does every defensive call appear in every context stratum?
coverage_df = check_positivity(
    df,
    treatment_col='def_playcall_enc',
    confounder_cols=['down', 'ydstogo', 'score_diff']
)

print(f"\nStrata with full positivity: {coverage_df['positivity_ok'].mean()*100:.1f}%")
print(f"\nWorst strata (most missing calls):")
coverage_df.sort_values('missing_calls', ascending=False).head(10)

---
## Stage 3a — Propensity Model

### What this stage does
Estimates $P(\text{DefPlaycall} = d \mid \text{Confounders})$ — the probability that each defensive call is chosen given the observed game context. This is the **propensity score** for a multi-class treatment.

### Theory: Why We Need Propensity Scores
In observational data, defensive coordinators don't call plays randomly — they react to formation, personnel, and tendencies. This means certain playcalls are systematically over- or under-represented in certain contexts, creating **confounding**. The propensity score captures this selection process.

We use it in two ways:
1. **IPW (Inverse Probability Weighting):** Reweight each play by $1/P(T \mid X)$ so that the weighted sample looks as if playcalls were chosen randomly. This "deconfounds" the observed data.
2. **AIPW (doubly robust):** Combine propensity scores with the outcome model to get an estimator that's consistent if *either* model is correctly specified.

### Cross-Fitting
Propensity scores are estimated via **cross-fitting** (out-of-fold predictions across 5 folds). This prevents the propensity model from overfitting to the same observations it will later reweight, which can cause "propensity score overfitting bias" — a well-documented failure mode when using flexible models like gradient boosting.

> **Edit here:** Tune `n_estimators`, `max_depth`, and `min_child_samples` in the LightGBM classifier to balance bias/variance. Increase `n_splits` for more stable cross-fit estimates at the cost of runtime.

In [ ]:
from pipeline_03_estimation import build_preprocessor, fit_propensity_model
from sklearn.preprocessing import LabelEncoder

# Build feature matrix
preprocessor = build_preprocessor()
X = preprocessor.fit_transform(df[ALL_CONFOUNDERS])
T = df[TREATMENT_COL].values.astype(int)
Y = df[OUTCOME_COL].values.astype(float)
n_calls = len(label_encoder.classes_)

feature_names = (
    list(preprocessor.named_transformers_['num'].get_feature_names_out())
    if hasattr(preprocessor.named_transformers_['num'], 'get_feature_names_out')
    else ALL_CONFOUNDERS[:X.shape[1]]
)

print(f"Feature matrix shape: {X.shape}")
print(f"Treatment classes: {n_calls}")

In [ ]:
# Fit the propensity model via cross-fitting
propensity = fit_propensity_model(X, T, n_splits=5)

# Inspect propensity score distributions — overlap is critical
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, min(n_calls, 4), figsize=(14, 3), sharey=True)
for i, ax in enumerate(axes):
    ax.hist(propensity[:, i], bins=40, color='steelblue', alpha=0.8, edgecolor='white')
    ax.set_title(f"{label_encoder.classes_[i]}\n(class {i})", fontsize=8)
    ax.set_xlabel('P(T=d | X)')
    ax.axvline(propensity[:, i].mean(), color='red', linestyle='--', label='mean')
plt.suptitle('Propensity Score Distributions per Defensive Call', y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nMean propensity per call:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls:<45} {propensity[:, i].mean():.3f}")

---
## Stage 3b — Outcome Model

### What this stage does
Estimates $E[\text{EPA} \mid \text{DefPlaycall} = d, \text{Confounders}]$ — the expected EPA for each defensive call, conditional on the full set of confounders. This is sometimes called the **Q-function** or **regression adjustment**.

### Theory: Outcome Regression
A direct approach to causal estimation is to fit a regression model and read off the effect by comparing predictions under different treatment values. If the model is correctly specified, this recovers the causal effect. In practice, flexible models (like gradient boosting) can approximate complex interactions between the defensive call and game context — for example, how a "heavy blitz" interacts with a shotgun formation.

We fit **one model per treatment level**, training only on plays where that call was made. This avoids forcing a single model to extrapolate across treatment levels, which can introduce bias when the relationship between confounders and outcome differs substantially between calls.

Cross-fitting is applied here as well — outcome models are trained on held-out folds to prevent the same overfitting issue described in Stage 3a.

> **Edit here:** If a particular treatment level has very few observations, consider increasing `min_child_samples` or lowering `n_estimators` for that level to prevent overfitting to sparse data.

In [ ]:
from pipeline_03_estimation import fit_outcome_model

# Fit outcome model for each treatment level via cross-fitting
mu_hat = fit_outcome_model(X, T, Y, n_treatment_classes=n_calls, n_splits=5)

print(f"Outcome model output shape: {mu_hat.shape}  (n_plays × n_calls)")
print("\nMean predicted EPA under each defensive call (outcome model):")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls:<45} {mu_hat[:, i].mean():+.4f}")

In [ ]:
# Inspect outcome model residuals for the observed treatment
n = len(Y)
mu_observed = mu_hat[np.arange(n), T]
residuals = Y - mu_observed

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(residuals, bins=60, color='coral', alpha=0.85, edgecolor='white')
ax1.axvline(0, color='black', linestyle='--')
ax1.set_title('Outcome Model Residuals')
ax1.set_xlabel('Residual EPA')

ax2.scatter(mu_observed[:2000], residuals[:2000], alpha=0.2, s=8, color='steelblue')
ax2.axhline(0, color='red', linestyle='--')
ax2.set_title('Predicted vs Residual (first 2k plays)')
ax2.set_xlabel('Predicted EPA')
ax2.set_ylabel('Residual')

plt.tight_layout()
plt.show()

print(f"Residual RMSE: {np.sqrt((residuals**2).mean()):.4f}")
print(f"Residual mean (should be ~0): {residuals.mean():.4f}")

---
## Stage 3c — AIPW Doubly Robust ATE Estimation

### What this stage does
Combines the propensity model and outcome model into an **Augmented Inverse Probability Weighted (AIPW)** estimator to produce the **Average Treatment Effect (ATE)** of each defensive call relative to a baseline.

### Theory: Doubly Robust Estimation
The AIPW estimator has the key property of **double robustness**: it produces a consistent estimate of the causal effect as long as *at least one* of the two component models (propensity or outcome) is correctly specified. This makes it significantly more reliable than either IPW or outcome regression alone.

The AIPW score for treatment level $d$ is:
$$\psi_d(i) = \hat{\mu}_d(X_i) + \frac{\mathbf{1}[T_i = d]}{\hat{P}(T=d \mid X_i)} \cdot \left(Y_i - \hat{\mu}_d(X_i)\right)$$

The ATE for call $d$ vs baseline $d_0$ is then simply:
$$\widehat{\text{ATE}}_{d \text{ vs } d_0} = \mathbb{E}[\psi_d] - \mathbb{E}[\psi_{d_0}]$$

**Interpretation:** A negative ATE means the defensive call reduces EPA compared to the baseline — i.e., it's better for the defense. A positive ATE means it allows more EPA.

> **Edit here:** Change `baseline` to use a different reference defensive call as your comparison point (e.g. the most common call in your dataset). The `baseline` argument takes an integer index into `label_encoder.classes_`.

In [ ]:
from pipeline_03_estimation import aipw_ate

# Estimate ATE via AIPW — baseline=0 uses the first treatment class as reference
# Change baseline= to any integer index to use a different reference call
ate_df = aipw_ate(
    Y, T, propensity, mu_hat,
    treatment_classes=list(label_encoder.classes_),
    baseline=0
)

print("ATE Estimates (negative = better for defense vs baseline)")
print("=" * 70)
print(ate_df.to_string(index=False))

In [ ]:
# Forest plot of ATE estimates with confidence intervals
ate_sorted = ate_df.sort_values('ate_vs_baseline')

fig, ax = plt.subplots(figsize=(10, max(4, len(ate_sorted) * 0.5 + 1)))

colors = ['#2ecc71' if v < 0 else '#e74c3c' for v in ate_sorted['ate_vs_baseline']]
y_pos = range(len(ate_sorted))

ax.barh(y_pos, ate_sorted['ate_vs_baseline'], color=colors, alpha=0.8, height=0.6)
ax.errorbar(
    ate_sorted['ate_vs_baseline'], y_pos,
    xerr=[
        ate_sorted['ate_vs_baseline'] - ate_sorted['ci_95_lo'],
        ate_sorted['ci_95_hi'] - ate_sorted['ate_vs_baseline']
    ],
    fmt='none', color='black', capsize=4, linewidth=1.5
)
ax.axvline(0, color='black', linewidth=1.2, linestyle='--')
ax.set_yticks(y_pos)
ax.set_yticklabels(ate_sorted['def_playcall'], fontsize=8)
ax.set_xlabel('ATE vs Baseline (EPA)')
ax.set_title('Causal Effect of Each Defensive Call on EPA\n(Green = reduces EPA, better for defense)', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Stage 3d — Causal Forest CATE Estimation

### What this stage does
Where Stage 3c estimates a single average effect across all plays, this stage estimates **heterogeneous treatment effects** — how the causal effect of each defensive call varies depending on the specific game context. This is called the **Conditional Average Treatment Effect (CATE)**.

### Theory: Causal Forests & the DR Learner
A **Causal Forest** (Wager & Athey, 2018) is an ensemble of causal trees, each trained to find partitions of the feature space where the treatment effect is most different. Unlike a standard random forest (which predicts outcomes), a causal forest directly targets $\tau(x) = E[Y(1) - Y(0) \mid X=x]$.

We use `econml`'s `ForestDRLearner`, which wraps the causal forest in the doubly robust framework from Stage 3c:
1. First, it orthogonalizes $Y$ and $T$ with respect to $X$ using the propensity and outcome models
2. Then, it fits a causal forest on the residualized quantities to estimate heterogeneous effects

This gives us a function $\hat{\tau}_d(x)$ — for any game state $x$, the estimated causal effect of defensive call $d$. This is the core of the agent's decision function in Stage 4.

> **Edit here:** Increase `n_estimators` (default 500) for more stable CATE estimates. Adjust `min_samples_leaf` to control granularity — smaller values allow more heterogeneity but risk overfitting.

In [ ]:
from pipeline_03_estimation import fit_causal_forest

# Fit the doubly robust causal forest
# This is the most computationally expensive step — ~5-15 min on 3 seasons of data
print("Fitting Causal Forest... (this may take several minutes)")
cf_model = fit_causal_forest(X, T, Y)

# Estimate CATEs for all plays in the dataset
cate_estimates = cf_model.effect(X)
print(f"\nCATE estimates shape: {cate_estimates.shape}  (n_plays × n_treatment_pairs)")
print(f"Mean CATE across all plays per treatment pair:")
print(np.round(cate_estimates.mean(axis=0), 4))

In [ ]:
# Visualize CATE distribution for the first treatment pair
# This shows how much effect heterogeneity exists across game contexts
cate_col = cate_estimates[:, 0] if cate_estimates.ndim > 1 else cate_estimates

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(cate_col, bins=60, color='mediumpurple', alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', linestyle='--', linewidth=1.5)
ax.axvline(cate_col.mean(), color='red', linestyle='-', linewidth=1.5, label=f'Mean={cate_col.mean():.3f}')
ax.set_xlabel('CATE (treatment effect on EPA)')
ax.set_title('Distribution of Conditional Treatment Effects (CATE)\nHeterogeneity across game contexts', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f"CATE std dev: {cate_col.std():.4f} — higher = more heterogeneity in the effect")
print(f"Plays where this call helps (CATE < 0): {(cate_col < 0).mean()*100:.1f}%")

In [ ]:
# Interpret the CATE via a shallow decision tree
# This extracts human-readable rules: "when X and Y, call Z reduces EPA by N"
from pipeline_03_estimation import interpret_cate_tree

all_feature_names = (
    list(ALL_CONFOUNDERS[:len([c for c in ALL_CONFOUNDERS if c not in ['formation','personnel_group']])])
    + list(preprocessor.named_transformers_['cat'].get_feature_names_out(['formation','personnel_group']))
)

# Plot the CATE interpretation tree for treatment index 1
# Change treatment_idx to explore different treatment pairs
interpret_cate_tree(cf_model, X, feature_names=all_feature_names, treatment_idx=1, max_depth=3)

---
## Stage 3e — Sensitivity Analysis

### What this stage does
Tests how robust the ATE estimates from Stage 3c are to **unmeasured confounding** — specifically the player quality confounders we couldn't fully observe (e.g., exact coverage grades, pass rush win rates).

### Theory: Rosenbaum Bounds
No causal analysis from observational data can fully rule out hidden confounders. Rosenbaum bounds (Rosenbaum, 2002) answer a specific question: *how strong would an omitted variable need to be to overturn our conclusions?*

The sensitivity parameter $\Gamma$ represents the odds ratio by which a hidden confounder could increase or decrease the probability of receiving a given treatment:
- $\Gamma = 1.0$: no hidden confounding (baseline assumption)
- $\Gamma = 2.0$: a hidden confounder could double the odds of receiving treatment

For each $\Gamma$, we compute worst-case bounds on the ATE. If the **sign** of our estimated effect remains stable across a wide range of $\Gamma$, we have strong evidence that the causal conclusion is robust — even without perfect player grades.

> **Edit here:** Extend `gamma_range` up to 3.0 or 4.0 if you want to stress-test further. Effects that only hold at $\Gamma = 1.0$ are fragile; effects that hold at $\Gamma \geq 2.0$ are considered robust.

In [ ]:
from pipeline_03_estimation import sensitivity_analysis

# Run Rosenbaum bounds sensitivity analysis
# gamma_range: how strong an unmeasured confounder we're testing against
sens_df = sensitivity_analysis(
    ate_df,
    gamma_range=[1.0, 1.25, 1.5, 1.75, 2.0, 2.5, 3.0]
)

print("Sensitivity Analysis — Sign Stability by Gamma")
print("(True = effect sign holds even under this level of hidden confounding)")
print()

pivot = sens_df.pivot_table(
    index='def_playcall', columns='gamma', values='sign_stable'
)
print(pivot.to_string())

In [ ]:
# Plot sensitivity bounds for the most impactful calls
top_calls = ate_df.nsmallest(4, 'ate_vs_baseline')['def_playcall'].tolist()

fig, axes = plt.subplots(1, len(top_calls), figsize=(14, 4), sharey=False)
if len(top_calls) == 1:
    axes = [axes]

for ax, call in zip(axes, top_calls):
    sub = sens_df[sens_df['def_playcall'] == call]
    ax.fill_between(sub['gamma'], sub['ate_lower_bound'], sub['ate_upper_bound'],
                    alpha=0.3, color='steelblue', label='Bounds')
    ax.plot(sub['gamma'], (sub['ate_lower_bound'] + sub['ate_upper_bound']) / 2,
            color='steelblue', linewidth=2, label='ATE')
    ax.axhline(0, color='red', linestyle='--', linewidth=1.2)
    ax.set_title(call, fontsize=7, fontweight='bold')
    ax.set_xlabel('Gamma (confounding strength)')
    ax.set_ylabel('ATE bounds')

plt.suptitle('Rosenbaum Sensitivity Bounds — Top Defensive Calls', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Stage 4 — Build the Defensive Coordinator Agent

### What this stage does
Packages all of the fitted models — the causal forest, preprocessor, label encoder, ATE estimates, and positivity coverage — into a `DefensiveCoordinatorAgent` that accepts a `GameState` and returns a ranked `PlaycallRecommendation`.

### How the Agent Makes Decisions
The agent's decision rule is:
$$\hat{d}^* = \underset{d}{\arg\min} \; E[\text{EPA} \mid do(\text{DefPlaycall} = d), \, X = x]$$

It estimates expected EPA for each call by adding the causal forest's CATE to the observed baseline EPA, then ranks calls from lowest (best for defense) to highest. Crucially, this marginalizes over the **distribution of offensive playcalls** via `off_pass_tendency` — so the agent is robust to offensive mixing rather than being exploitable by a single counterplay.

The agent also uses the positivity check from Stage 2 to flag low-confidence situations. If the game context falls in a stratum where some defensive calls are rarely observed in the training data, confidence is downgraded to `"low"` to prevent the agent from making overconfident extrapolations into uncharted territory.

> **Edit here:** The `GameState` dataclass in Stage 5 is the primary interface for the agent. Fields marked with defaults can be left as-is; fields with no defaults (down, ydstogo, etc.) must be provided for each query.

In [ ]:
from pipeline_04_agent import DefensiveCoordinatorAgent, GameState, PlaycallRecommendation

# Assemble the agent from all fitted components
agent = DefensiveCoordinatorAgent(
    causal_forest=cf_model,
    preprocessor=preprocessor,
    label_encoder=label_encoder,
    ate_df=ate_df,
    coverage_df=coverage_df,
    baseline_epa=float(Y.mean()),
)

print("Agent built successfully.")
print(f"  Defensive calls available: {len(label_encoder.classes_)}")
print(f"  Positivity-covered strata: {coverage_df['positivity_ok'].mean()*100:.1f}%")
print(f"  Baseline EPA (league avg):  {Y.mean():.4f}")

---
## Stage 5 — Playcall Recommendations

### What this stage does
Runs the agent on concrete game scenarios and displays its recommendations. Each recommendation includes the optimal defensive call, its expected EPA with a 95% confidence interval, the top 3 alternatives, a confidence rating, and a plain-English reasoning string drawn from the game state features.

Three scenarios are provided as starting points:
- **Scenario 1:** 3rd & 8, shotgun, pass-heavy tendency — classic passing situation
- **Scenario 2:** 1st & 10, singleback, balanced tendency — standard early down
- **Scenario 3:** 2nd & 3, two-minute drill, red zone, run-heavy formation — goal-line pressure

> **Edit here:** Modify the `GameState(...)` fields in any scenario to test the agent against your own game situations. All scenarios run independently — you can add as many as you like.

In [ ]:
# ── Scenario 1: 3rd & 8 — Classic passing situation ──────────────────────────
scenario_1 = GameState(
    down=3,
    ydstogo=8,
    score_diff=-3,               # defense is winning by 3
    seconds_remaining=420,       # 7 minutes left
    yardline_100=65,             # own 35-yard line
    formation="SHOTGUN",
    num_rb=1,
    num_te=1,
    num_wr=3,
    off_pass_tendency=0.82,      # this team passes 82% here
    def_team_avg_epa=-0.04,      # above-average defense
    qb_avg_epa=0.15,             # above-average QB
)

rec1 = agent.recommend(scenario_1)
rec1.display()

In [ ]:
# ── Scenario 2: 1st & 10 — Standard early down, balanced offense ─────────────
scenario_2 = GameState(
    down=1,
    ydstogo=10,
    score_diff=7,                # offense is down 7
    seconds_remaining=1800,      # 2nd quarter
    yardline_100=75,             # own 25-yard line
    formation="SINGLEBACK",
    num_rb=2,
    num_te=1,
    num_wr=2,
    off_pass_tendency=0.48,      # balanced tendency
    def_team_avg_epa=-0.02,
    qb_avg_epa=0.05,
)

rec2 = agent.recommend(scenario_2)
rec2.display()

In [ ]:
# ── Scenario 3: 2nd & 3, red zone, two-minute drill ──────────────────────────
scenario_3 = GameState(
    down=2,
    ydstogo=3,
    score_diff=0,                # tied game
    seconds_remaining=90,        # two-minute drill
    yardline_100=15,             # red zone — 15 yards from end zone
    formation="I_FORM",
    num_rb=2,
    num_te=2,
    num_wr=1,
    off_pass_tendency=0.38,      # run-heavy tendency
    def_team_avg_epa=0.01,       # slightly below-average defense
    qb_avg_epa=0.10,
)

rec3 = agent.recommend(scenario_3)
rec3.display()

In [ ]:
# ── Custom Scenario — edit this cell to test your own game situation ──────────
custom_scenario = GameState(
    down=4,                      # ← edit
    ydstogo=1,
    score_diff=-6,
    seconds_remaining=60,
    yardline_100=5,              # goal-line
    formation="I_FORM",
    num_rb=2,
    num_te=2,
    num_wr=1,
    off_pass_tendency=0.30,
    def_team_avg_epa=-0.03,
    qb_avg_epa=0.08,
)

rec_custom = agent.recommend(custom_scenario)
rec_custom.display()

In [ ]:
# ── Save pipeline outputs ─────────────────────────────────────────────────────
ate_df.to_csv('ate_estimates.csv', index=False)
sens_df.to_csv('sensitivity_analysis.csv', index=False)
coverage_df.to_csv('positivity_coverage.csv', index=False)

print("Outputs saved:")
print("  ate_estimates.csv       — AIPW doubly robust ATE estimates per defensive call")
print("  sensitivity_analysis.csv — Rosenbaum bounds at each gamma level")
print("  positivity_coverage.csv  — Strata coverage for positivity diagnostics")
print("  dag.png                  — Causal DAG visualization")